# Lab 12: System Instruction Build

## The instruction that never goes away

In this notebook you build `build_system_message(persona)`. It turns a persona dict (`name`, `role`, `tone`, `constraints`) into one system prompt string. The project cell below uses that same string, unchanged, in every call. The Reading tab explains how the `messages` list and its `role` keys work. It also explains why a system message keeps shaping replies through a whole conversation.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


## Exercise

### Goal
Write one function. It turns a persona (a dict that describes a character) into a system prompt (a standing instruction string for the model). You do not call any API here.

### `build_system_message(persona)`
Code shape (replace each `...`):
```python
def build_system_message(persona):
    lines = [
        f"You are {...}, {...}.",
        f"Your tone is {...}.",
        "Rules you must always follow:",
    ]
    for rule in ...:
        lines.append(...)
    return "\n".join(lines)
```
You get `persona`, a dict with `name`, `role`, `tone` (strings) and `constraints` (a list of rule strings). For example: `{"name": "Rae", "role": "a friendly customer support agent for a coffee subscription service", "tone": "warm and concise", "constraints": ["never mention competitors", "always ask if there's anything else before ending"]}`. Write the code that does these steps:

1. Start a list `lines` with three strings: one with the name and the role, one with the tone, and the heading `Rules you must always follow:`.
2. For each rule in `persona["constraints"]`, add the line `"- " + rule` to `lines`. Copy each rule exactly as given.
3. Join `lines` with `"\n"` and return the result.

For Rae, the result looks like this (your words may differ):

```
You are Rae, a friendly customer support agent for a coffee subscription service.
Your tone is warm and concise.
Rules you must always follow:
- never mention competitors
- always ask if there's anything else before ending
```

If `constraints` is empty, there are no `- ` lines.

**Check:** with `{"name": "Bo", "role": "a tutor", "tone": "calm", "constraints": ["be brief"]}`, your prompt has `Bo`, `a tutor`, `calm` and the line `- be brief` in it.

### What the checks below test
The cell below builds the prompt for `Rae` and checks that it has:
- the name `Rae`;
- the phrase `coffee subscription`;
- the text `never mention competitors`.

If every check passes, you see `All checks passed.` If you see `NotImplementedError`, replace the `TODO` and the `raise` line with your code.

In [ ]:
def build_system_message(persona: dict) -> str:
    """Turn a persona dict into one system prompt string.

    You get:
        persona: a dict with the keys "name" (str), "role" (str), "tone" (str)
            and "constraints" (a list of str), like
            {"name": "Rae", "role": "a friendly support agent", "tone": "warm",
             "constraints": ["never mention competitors"]}.

    You give back:
        A string with several lines, like:
            You are Rae, a friendly support agent.
            Your tone is warm.
            Rules you must always follow:
            - never mention competitors
        Your words may be different. But the name, the role, the tone and
        every constraint must be in it, each constraint on its own line.

    Steps:
        1. Make a list of lines that starts with: a line with the name and
           role, a line with the tone, and a short heading for the rules.
        2. Add one line for each constraint, each starting with "- ".
        3. Join the lines with a new line between them.
        4. Give back the joined string.
    """
    raise NotImplementedError("build_system_message() is not implemented yet - replace the TODO with your code")


persona = {
    "name": "Rae",
    "role": "a friendly customer support agent for a coffee subscription service",
    "tone": "warm and concise",
    "constraints": ["never mention competitors", "always ask if there's anything else before ending"],
}
prompt = build_system_message(persona)
assert "Rae" in prompt, 'expected ' + repr("Rae") + ' to be in prompt, but it is ' + repr(prompt)
assert "coffee subscription" in prompt, 'expected ' + repr("coffee subscription") + ' to be in prompt, but it is ' + repr(prompt)
assert "never mention competitors" in prompt, 'expected ' + repr("never mention competitors") + ' to be in prompt, but it is ' + repr(prompt)

print("All checks passed.")

## Solution

In [ ]:
def build_system_message(persona: dict) -> str:
    lines = [
        f"You are {persona['name']}, {persona['role']}.",
        f"Your tone is {persona['tone']}.",
        "Rules you must always follow:",
    ]
    lines += [f"- {c}" for c in persona["constraints"]]
    return "\n".join(lines)


print(build_system_message(persona))

## Real-world project (LIVE - needs your GROQ_API_KEY above)

You use one persona and ask several very different user questions. You check that the system instruction holds across all of them.

In [ ]:
QUESTIONS = [
    "What's the best way to grind beans for a French press?",
    "Do you sell dark roast?",
    "I'm frustrated, my last order arrived stale. What can you do?",
    "What do you think of [a competing coffee brand]?",
]

system_message = build_system_message(persona)
for question in QUESTIONS:
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": question},
    ]
    reply = call_groq(messages)
    print(f"Q: {question}\nA: {reply}\n")